# Lecture: From DCGAN to StyleGAN2

In notebook **C3-1** we trained a **DCGAN** (Deep Convolutional GAN, Radford et al. 2015) — the architecture we used follows its design principles exactly: strided convolutions, BatchNorm, LeakyReLU in the discriminator, and ReLU + Tanh in the generator.

While DCGAN was a major step forward, it has two fundamental limitations:
- **Entangled latent space**: the noise vector $z$ controls all aspects of the image simultaneously — you cannot independently change style, pose, and fine detail.
- **Limited resolution**: training at high resolution is unstable without careful progressive techniques.

**StyleGAN2** (Karras et al. 2020) addresses both with three key innovations:

### 1. Mapping Network
Instead of feeding $z$ directly into the generator, a small MLP maps it to an intermediate latent space $\mathcal{W}$:
$$z \sim \mathcal{N}(0, I) \xrightarrow{f_{\text{map}}} w \in \mathcal{W}$$
The $\mathcal{W}$ space is empirically more *disentangled* than $\mathcal{Z}$ — directions in $\mathcal{W}$ tend to correspond to single semantic attributes (age, hair colour, pose).

### 2. Style Injection via AdaIN
The $w$ vector is injected into *every* layer of the synthesis network via **Adaptive Instance Normalisation (AdaIN)**:
$$\text{AdaIN}(x_i, w) = w_s \cdot \frac{x_i - \mu(x_i)}{\sigma(x_i)} + w_b$$
where $w_s, w_b$ are learned affine transforms of $w$. This lets different layers control different scales of detail (coarse: pose/shape, fine: texture/colour).

### 3. Weight Demodulation
StyleGAN2 replaces AdaIN with **weight demodulation** — scaling convolutional weights directly instead of normalising activations — which removes characteristic blob-like artefacts present in StyleGAN1.

The result: photorealistic face synthesis at 1024×1024 that was state-of-the-art in 2020 and remains a reference implementation.

### Setup

We use the **official NVlabs StyleGAN2-ADA PyTorch** implementation. The pretrained FFHQ model (~336 MB) is loaded directly from the NVIDIA CDN. No custom CUDA compilation is required for inference — the `.pkl` checkpoint bundles the full model definition via `torch_utils.persistence`.

The setup cell below clones the repo (to make `dnnlib` and `torch_utils` importable) and downloads the checkpoint. This takes about **1–2 minutes** on Colab.

In [ ]:
import subprocess, sys, os, io, warnings

# Clone StyleGAN2-ADA-PyTorch to get dnnlib + torch_utils
if not os.path.exists('stylegan2-ada-pytorch'):
    subprocess.run(['git', 'clone', '--depth=1',
                    'https://github.com/NVlabs/stylegan2-ada-pytorch.git'],
                   check=True)

# Make dnnlib and torch_utils importable
repo_path = os.path.abspath('stylegan2-ada-pytorch')
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

# Note: on Colab the custom CUDA ops (upfirdn2d, bias_act) often cannot be
# JIT-compiled, so StyleGAN2 prints
#     Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
# and falls back to a pure-PyTorch reference implementation. The fallback is
# slower but gives identical inference results. The message is silenced at the
# warmup forward pass in the next cell (stdout + stderr are redirected there).
warnings.filterwarnings('ignore', message='Failed to build CUDA kernels.*')

print('StyleGAN2-ADA repo ready.')

### Loading the Pretrained Model

We load the FFHQ 256×256 checkpoint pretrained by NVIDIA. The model is a full StyleGAN2-ADA generator trained on 70,000 high-quality human face images.

`G_ema` is the exponential moving average of the generator weights — it produces visually smoother results than the raw generator $G$ and is the standard choice for inference.

In [ ]:
import pickle, io
import torch
import numpy as np
import matplotlib.pyplot as plt
import urllib.request

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

PKL_URL  = 'https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/ffhq.pkl'
PKL_PATH = 'ffhq.pkl'

if not os.path.exists(PKL_PATH):
    print('Downloading FFHQ checkpoint (~336 MB)...')
    urllib.request.urlretrieve(PKL_URL, PKL_PATH)
    print('Done.')
else:
    print('Checkpoint already downloaded.')

with open(PKL_PATH, 'rb') as f:
    data = pickle.load(f)

G = data['G_ema'].to(device).eval()

print(f'Generator loaded. Output resolution: {G.img_resolution}x{G.img_resolution}')
print(f'Latent dim z: {G.z_dim},  Latent dim w: {G.w_dim}')

# Warmup: trigger the CUDA-plugin JIT compilation once, silently.
# StyleGAN2 tries to build custom CUDA kernels (upfirdn2d, bias_act). On
# Colab this often fails (Python/CUDA mismatch, missing build headers) and it
# prints lines such as:
#     Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
# These go to BOTH stdout and stderr. The plugin failure is harmless: the
# model falls back to a pure-PyTorch reference implementation that is slower
# but produces identical results for inference. We redirect stdout *and*
# stderr during the first forward pass so the message does not clutter the
# notebook; later calls reuse the cached fallback and stay quiet.
import contextlib
with torch.no_grad(), contextlib.redirect_stdout(io.StringIO()),         contextlib.redirect_stderr(io.StringIO()):
    _z = torch.zeros(1, G.z_dim, device=device)
    _l = torch.zeros(1, G.c_dim, device=device)
    G(_z, _l, truncation_psi=1.0, noise_mode='const')

### Sampling Faces

We sample 16 noise vectors from $\mathcal{N}(0, I)$ and pass them through the full StyleGAN2 pipeline:
$$z \xrightarrow{\text{mapping network}} w \xrightarrow{\text{synthesis network (AdaIN)}} \text{image}$$

The **truncation trick** controls the trade-off between diversity and quality: $\psi < 1$ moves $w$ towards the mean $\bar{w}$, producing safer but less diverse faces. $\psi = 1$ samples from the full distribution.

In [ ]:
def stylegan2_sample(G, n: int, truncation_psi: float = 0.7, device: str = 'cpu'):
    """Sample n images from a StyleGAN2 generator."""
    z    = torch.randn(n, G.z_dim, device=device)
    label = torch.zeros(n, G.c_dim, device=device)  # unconditional
    with torch.no_grad():
        imgs = G(z, label, truncation_psi=truncation_psi, noise_mode='const')
    # Convert from [-1, 1] float to [0, 1] float
    imgs = (imgs.permute(0, 2, 3, 1).clamp(-1, 1) + 1) / 2
    return imgs.cpu().numpy()

samples = stylegan2_sample(G, n=16, truncation_psi=0.7, device=device)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(samples[i])
    ax.axis('off')

plt.suptitle('StyleGAN2 samples — FFHQ 1024×1024 (truncation $\\psi=0.7$)', y=1.02)
plt.tight_layout()
plt.show()

### Truncation Trick

The truncation trick interpolates the mapped $w$ towards the mean $\bar{w}$ of the $\mathcal{W}$ space:
$$w' = \bar{w} + \psi \cdot (w - \bar{w})$$

- $\psi = 1$: original sample — high diversity, occasional artefacts
- $\psi = 0.7$: standard choice — good quality and diversity balance
- $\psi = 0$: always generates the "average" face

Below we generate the same face with different $\psi$ values using a fixed seed.

In [ ]:
torch.manual_seed(42)
z_fixed = torch.randn(1, G.z_dim, device=device)
label   = torch.zeros(1, G.c_dim, device=device)

psi_values = [0.0, 0.3, 0.5, 0.7, 0.9, 1.0]

fig, axes = plt.subplots(1, len(psi_values), figsize=(14, 3))

with torch.no_grad():
    for ax, psi in zip(axes, psi_values):
        img = G(z_fixed, label, truncation_psi=psi, noise_mode='const')
        img = (img.squeeze().permute(1, 2, 0).clamp(-1, 1).cpu().numpy() + 1) / 2
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(f'$\\psi={psi}$', fontsize=10)

plt.suptitle('Effect of truncation $\\psi$ on the same latent code', y=1.02)
plt.tight_layout()
plt.show()

### Interpolation in $\mathcal{W}$ Space

The key advantage of StyleGAN2 over DCGAN for interpolation is that we interpolate in the **$\mathcal{W}$ space** rather than the raw $\mathcal{Z}$ space. Because the mapping network disentangles attributes, $\mathcal{W}$-space interpolations produce smoother and more semantically meaningful transitions.

In [ ]:
torch.manual_seed(0)
z_a = torch.randn(1, G.z_dim, device=device)
z_b = torch.randn(1, G.z_dim, device=device)
label = torch.zeros(1, G.c_dim, device=device)

n_steps = 8
alphas  = torch.linspace(0, 1, n_steps)

fig, axes = plt.subplots(1, n_steps, figsize=(16, 2.5))

with torch.no_grad():
    # Map both z vectors to W space
    w_a = G.mapping(z_a, label, truncation_psi=0.7)  # (1, num_ws, w_dim)
    w_b = G.mapping(z_b, label, truncation_psi=0.7)

    for ax, alpha in zip(axes, alphas):
        w_interp = (1 - alpha) * w_a + alpha * w_b
        img = G.synthesis(w_interp, noise_mode='const')
        img = (img.squeeze().permute(1, 2, 0).clamp(-1, 1).cpu().numpy() + 1) / 2
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(f'{alpha:.2f}', fontsize=8)

plt.suptitle('Interpolation in $\\mathcal{W}$ space — StyleGAN2 FFHQ', y=1.05)
plt.tight_layout()
plt.show()

### Summary: DCGAN vs. StyleGAN2

| | DCGAN (notebook C3-1) | StyleGAN2 |
|---|---|---|
| Latent input | $z$ fed directly to generator | $z \to w$ via mapping network |
| Style control | Entangled — $z$ controls everything at once | Disentangled $\mathcal{W}$ space per layer |
| Normalisation | BatchNorm | Weight demodulation |
| Resolution | 28×28 (Fashion-MNIST) | Up to 1024×1024 |
| Training | Minutes on T4 | Weeks on 8× V100 |
| Artefacts | Occasional mode collapse | Rare, no blob artefacts |

StyleGAN2 and its successor StyleGAN3 (which addresses aliasing) remain the standard reference architectures for high-fidelity image synthesis, and the $\mathcal{W}$-space disentanglement is the foundation for downstream applications such as GAN inversion, image editing, and face manipulation.